# Graph-SJEPA Pretraining for Cross-Dataset Motor Imagery EEG

This notebook is a first **Graph-SJEPA pretraining** implementation designed to extend the S-JEPA workflow toward cross-dataset MI transfer.

The core change from standard S-JEPA is the spatial pathway:

```text
standard S-JEPA:
  EEG channels → local encoder → contextual encoder → predictor

Graph-SJEPA:
  EEG channels → local encoder → graph spatial encoder → contextual encoder → predictor
```

Why this exists:

- Lee2019, PhysioNetMI, Cho2017, BNCI2014_001, Weibo2014, and other MI datasets have different channel counts and montage layouts.
- A fixed spatial convolution does not naturally transfer across those datasets.
- This notebook uses **electrode coordinates** to build a graph per dataset/chunk, so the model can process variable channel layouts.

Pretraining remains self-supervised and JEPA-style:

1. Encode EEG windows into latent tokens.
2. Mask a spatial block of channels.
3. Let the student see masked/zeroed channels.
4. Let the EMA teacher see the full signal.
5. Predict teacher latent targets only at masked tokens.
6. Save the best student backbone for downstream MI fine-tuning.

This is meant to be the pretraining counterpart to a future downstream model named something like:

```text
Graph-SJEPA downstream classifier
```

# 1. Setup

## 1.1. Import Libraries

In [1]:
import sys
from pathlib import Path
import platform
import json
import hashlib
from datetime import datetime
import builtins
import random
import gc
import time
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import mne
mne.set_log_level("WARNING")

from braindecode.datasets import MOABBDataset
from braindecode.datasets.base import BaseConcatDataset
from braindecode.preprocessing import Preprocessor, preprocess
from braindecode.preprocessing.windowers import create_fixed_length_windows

import matplotlib.pyplot as plt

print("Imports loaded successfully.")

/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports loaded successfully.


## 1.2. Runtime & Path Validation

In [2]:
print("Runtime Environment:")
print(f"  - Python:   {sys.version}")
print(f"  - Platform: {platform.platform()}")
print(f"  - Torch:    {torch.__version__}")
print(f"  - CUDA:     {torch.cuda.is_available()}")

WORKING_DIR = Path.cwd().parent
print(f"\nWorking directory: {WORKING_DIR}")

Runtime Environment:
  - Python:   3.11.15 (main, Apr  9 2026, 01:18:52) [Clang 21.0.0 (clang-2100.0.123.102)]
  - Platform: macOS-26.2-arm64-arm-64bit
  - Torch:    2.10.0
  - CUDA:     False

Working directory: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA


# 2. Configuration

In [ ]:
# Dataset defaults for MOABB motor-imagery transfer experiments.
MOABB_MI_DATASET_CONFIGS = {
    "Lee2019_MI": {
        # https://moabb.neurotechx.com/docs/generated/moabb.datasets.Lee2019_MI.html
        "goal": "same-dataset reproduction for Lee2019-pretrained S-JEPA",
        # "default_subjects": list(range(48, 55)),
        "default_subjects": list(range(48, 55)),
        "exclude_subjects": [],
        "labels_to_keep": ["left_hand", "right_hand"],
        "moabb_dataset_kwargs": {},
    },
    "Cho2017": {
        # https://moabb.neurotechx.com/docs/generated/moabb.datasets.Cho2017.html
        "goal": "closest external high-density binary left/right MI transfer test",
        # "default_subjects": list(range(1, 53)),
        "default_subjects": list(range(1, 10)),
        "exclude_subjects": [],
        "labels_to_keep": ["left_hand", "right_hand"],
        "moabb_dataset_kwargs": {},
    },
    "Stieger2021": {
        # https://moabb.neurotechx.com/docs/generated/moabb.datasets.Stieger2021.html
        "goal": "same 62-channel scale, more sessions, left/right subset only",
        # "default_subjects": list(range(1, 63)),
        "default_subjects": list(range(1, 10)),
        "exclude_subjects": [],
        "labels_to_keep": ["left_hand", "right_hand"],
        "moabb_dataset_kwargs": {},
    },
    "PhysionetMI": {
        # https://moabb.neurotechx.com/docs/generated/moabb.datasets.PhysionetMI.html
        "goal": "hard cross-dataset stress test",
        # "default_subjects": list(range(1, 110)),
        "default_subjects": list(range(1, 10)),
        "exclude_subjects": [88],
        "labels_to_keep": ["left_hand", "right_hand"],
        "moabb_dataset_kwargs": {"imagined": True, "executed": False},
    },
    "BNCI2014_001": {
        # https://moabb.neurotechx.com/docs/generated/moabb.datasets.BNCI2014_001.html
        "goal": "standard BCI-IV-2a benchmark, left/right subset only",
        # "default_subjects": list(range(1, 10)),
        "default_subjects": list(range(1, 10)),
        "exclude_subjects": [],
        "labels_to_keep": ["left_hand", "right_hand"],
        "moabb_dataset_kwargs": {},
    },
    "Weibo2014": {
        # https://moabb.neurotechx.com/docs/generated/moabb.datasets.Weibo2014.html
        "goal": "broader benchmark comparison, left/right subset only",
        # "default_subjects": list(range(1, 11)),
        "default_subjects": list(range(1, 10)),
        "exclude_subjects": [],
        "labels_to_keep": ["left_hand", "right_hand"],
        "moabb_dataset_kwargs": {},
    },
}

## 2.1. Config

In [ ]:
CONFIG = {
    # Paths
    "artifact_dir": str(WORKING_DIR / "artifacts" / "graph-sjepa-mi-pretraining"),

    # Reproducibility
    "seed": 42,

    # Cross-dataset MI pretraining pool.
    # Keep this small first. Scale only after the notebook passes the sanity checks.
    "datasets": [
        {
            "name": "Lee2019_MI",
            "role": "source_high_density",
            "train_subject_ids": list(range(1, 6)),      # expand to list(range(1, 41)) later
            "val_subject_ids": list(range(41, 43)),      # expand to list(range(41, 48)) later
            "exclude_subject_ids": list(range(48, 55)),
            "dataset_kwargs": {},
        },
        {
            "name": "PhysionetMI",
            "role": "target_hard_transfer",
            "train_subject_ids": list(range(1, 6)),      # expand later
            "val_subject_ids": list(range(91, 94)),      # held-out subjects for validation
            "exclude_subject_ids": [88],
            "dataset_kwargs": {"imagined": True, "executed": False},
        },
    ],

    # Chunked data streaming.
    # One chunk contains one dataset + a small list of subjects. This avoids batching different channel counts together.
    "subject_chunk_size": 2,
    "chunk_shuffle": True,
    "window_preload": False,

    # Preprocessing. Keep aligned with S-JEPA pipeline.
    "sfreq": 128,
    "bandpass_low": 0.5,
    "bandpass_high": 40.0,

    # Fixed-window SSL pretraining.
    # Start with 4s for quicker debugging. Use 16s once everything works.
    "pretrain_duration_s": 4.0,      # paper-style options: 1.0, 4.0, 16.0
    "sampling_interval_s": 4.0,      # use 16.0 for non-overlapping 16s windows

    # Graph construction.
    "graph_k_neighbors": 6,
    "graph_self_loops": True,
    "graph_distance_weighted": True,
    "graph_eps": 1e-6,

    # Spatial block masking.
    "masking_strategy": "coordinate_spatial_block",
    "mask_diameter_percent": 60,     # paper-style: 40, 60, 80
    "min_visible_channels": 2,

    # Model architecture.
    "emb_dim": 64,
    "local_conv_channels": [32, 64, 64],
    "local_kernel_size": 7,
    "local_stride": 2,
    "graph_layers": 2,
    "context_layers": 4,
    "context_nhead": 8,
    "context_dim_feedforward": 256,
    "predictor_layers": 2,
    "predictor_nhead": 8,
    "predictor_dim_feedforward": 256,
    "dropout": 0.1,

    # Training.
    "batch_size": 16,
    "learning_rate": 1e-3,
    "weight_decay": 0.0,
    "n_epochs": 100,
    "ema_decay": 0.996,
    "early_stopping_patience": 10,
    "grad_clip_norm": 1.0,

    # Device: "auto" | "cpu" | "cuda" | "mps"
    "device": "auto",
}

## 2.2. Create Artifact Directory

In [4]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run ID: {RUN_ID}")
print(f"Artifact directory: {ARTIFACT_DIR}")

Run ID: 20260505_2148_2d1f6de5
Artifact directory: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/graph-sjepa-mi-pretraining/20260505_2148_2d1f6de5


## 2.3. Initialize Logger

In [ ]:
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1)


def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)

    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            sys.__stdout__.write(text)  # type: ignore
            if flush:
                sys.__stdout__.flush()  # type: ignore
        else:
            file.write(text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _LOG_FILE_HANDLE.write(blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _LOG_FILE_HANDLE.write(stamped + end)
    else:
        _write_target(end)
        _LOG_FILE_HANDLE.write(end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

print(f"Logging to: {LOG_PATH}")

[2026-05-05 21:48:30] Logging to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/graph-sjepa-mi-pretraining/20260505_2148_2d1f6de5/run.log


## 2.4. Device Configuration

In [6]:
def resolve_device(device):
    requested = str(device).lower()
    if requested == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    if requested == "cuda" and not torch.cuda.is_available():
        print("CUDA requested but unavailable; using CPU.")
        return torch.device("cpu")
    if requested == "mps" and not (hasattr(torch.backends, "mps") and torch.backends.mps.is_available()):
        print("MPS requested but unavailable; using CPU.")
        return torch.device("cpu")
    return torch.device(requested)

DEVICE = resolve_device(CONFIG["device"])
print(f"Using device: {DEVICE}")

[2026-05-05 21:48:30] Using device: mps


## 2.5. Deterministic Seeding

In [7]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
print(f"Seed set to: {CONFIG['seed']}")

[2026-05-05 21:48:30] Seed set to: 42


## 2.6. Save Configuration

In [8]:
config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as output_file:
    json.dump(CONFIG, output_file, indent=2, default=str)
print(f"Config saved to: {config_path}")

[2026-05-05 21:48:30] Config saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/graph-sjepa-mi-pretraining/20260505_2148_2d1f6de5/config.json


# 3. Data

## 3.1. Dataset Registry and Subject Chunks

In [9]:
def clean_subject_list(subject_ids, exclude_subject_ids):
    exclude = {int(s) for s in exclude_subject_ids}
    return [int(s) for s in subject_ids if int(s) not in exclude]


def iter_subject_chunks(subject_ids, chunk_size, shuffle=False, seed=None):
    if chunk_size <= 0:
        raise ValueError(f"chunk_size must be > 0, got {chunk_size}")
    ordered = [int(s) for s in subject_ids]
    if shuffle:
        rng = random.Random(seed)
        rng.shuffle(ordered)
    for start in range(0, len(ordered), chunk_size):
        chunk = ordered[start:start + chunk_size]
        if chunk:
            yield chunk


def iter_dataset_subject_chunks(split_name, shuffle=False, seed=None):
    """Yield dicts with dataset config + subject chunk for train or validation."""
    all_chunks = []
    for ds_cfg in CONFIG["datasets"]:
        subject_key = "train_subject_ids" if split_name == "train" else "val_subject_ids"
        subject_ids = clean_subject_list(
            ds_cfg.get(subject_key, []),
            ds_cfg.get("exclude_subject_ids", []),
        )
        for subject_chunk in iter_subject_chunks(
            subject_ids,
            chunk_size=CONFIG["subject_chunk_size"],
            shuffle=False,
        ):
            all_chunks.append({
                "dataset_name": ds_cfg["name"],
                "dataset_role": ds_cfg.get("role", "unknown"),
                "dataset_kwargs": dict(ds_cfg.get("dataset_kwargs", {})),
                "subject_ids": subject_chunk,
            })

    if shuffle:
        rng = random.Random(seed)
        rng.shuffle(all_chunks)
    return all_chunks


TRAIN_CHUNKS_PREVIEW = iter_dataset_subject_chunks("train", shuffle=False)
VAL_CHUNKS_PREVIEW = iter_dataset_subject_chunks("val", shuffle=False)

print("Graph-SJEPA dataset pool:")
for ds_cfg in CONFIG["datasets"]:
    train_subjects = clean_subject_list(ds_cfg["train_subject_ids"], ds_cfg.get("exclude_subject_ids", []))
    val_subjects = clean_subject_list(ds_cfg["val_subject_ids"], ds_cfg.get("exclude_subject_ids", []))
    print(f"  {ds_cfg['name']}: role={ds_cfg.get('role')} train={train_subjects} val={val_subjects} kwargs={ds_cfg.get('dataset_kwargs', {})}")
print(f"\nTrain chunks per epoch: {len(TRAIN_CHUNKS_PREVIEW)}")
print(f"Val chunks per epoch:   {len(VAL_CHUNKS_PREVIEW)}")

[2026-05-05 21:48:30] Graph-SJEPA dataset pool:
[2026-05-05 21:48:30]   Lee2019_MI: role=source_high_density train=[1, 2, 3, 4, 5] val=[41, 42] kwargs={}
[2026-05-05 21:48:30]   PhysionetMI: role=target_hard_transfer train=[1, 2, 3, 4, 5] val=[91, 92, 93] kwargs={'imagined': True, 'executed': False}

[2026-05-05 21:48:30] Train chunks per epoch: 6
[2026-05-05 21:48:30] Val chunks per epoch:   3


## 3.2. Pretraining Window Parameters

In [10]:
def compute_pretraining_window_params(sfreq, pretrain_duration_s, sampling_interval_s):
    if pretrain_duration_s not in {1.0, 4.0, 16.0}:
        print(
            "Warning: pretrain_duration_s is not one of the S-JEPA paper values "
            "{1, 4, 16}. This is allowed, but token geometry may differ."
        )
    window_size_samples = int(round(float(pretrain_duration_s) * float(sfreq)))
    sampling_interval_samples = int(round(float(sampling_interval_s) * float(sfreq)))
    if window_size_samples <= 0 or sampling_interval_samples <= 0:
        raise ValueError("Window and stride samples must be positive.")
    print("Pretraining window parameters:")
    print(f"  sfreq:                      {sfreq} Hz")
    print(f"  pretrain_duration_s:        {pretrain_duration_s} s")
    print(f"  sampling_interval_s:        {sampling_interval_s} s")
    print(f"  window_size_samples:        {window_size_samples}")
    print(f"  sampling_interval_samples:  {sampling_interval_samples}")
    return window_size_samples, sampling_interval_samples

WINDOW_SIZE_SAMPLES, SAMPLING_INTERVAL_SAMPLES = compute_pretraining_window_params(
    sfreq=CONFIG["sfreq"],
    pretrain_duration_s=CONFIG["pretrain_duration_s"],
    sampling_interval_s=CONFIG["sampling_interval_s"],
)

[2026-05-05 21:48:30] Pretraining window parameters:
[2026-05-05 21:48:30]   sfreq:                      128 Hz
[2026-05-05 21:48:30]   pretrain_duration_s:        4.0 s
[2026-05-05 21:48:30]   sampling_interval_s:        4.0 s
[2026-05-05 21:48:30]   window_size_samples:        512
[2026-05-05 21:48:30]   sampling_interval_samples:  512


## 3.3. Preprocessing

In [ ]:
def scale_volts_to_microvolts(data):
    return data * 1e6

def build_preprocessors(sfreq, bandpass_low, bandpass_high):
    preprocessors = [
        Preprocessor("pick_types", eeg=True, meg=False, stim=False),
        Preprocessor("set_eeg_reference", ref_channels="average"),
        Preprocessor(
            "filter",
            l_freq=bandpass_low,
            h_freq=bandpass_high,
        ),
        Preprocessor("resample", sfreq=sfreq),
        Preprocessor(scale_volts_to_microvolts)
    ]
    return preprocessors


_PREPROCESSORS = build_preprocessors(
    sfreq=CONFIG["sfreq"],
    bandpass_low=CONFIG["bandpass_low"],
    bandpass_high=CONFIG["bandpass_high"],
)

[2026-05-05 21:48:30] Number of preprocessors: 5


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:77: UserWarning: apply_on_array can only be True if fn is a callable function. Automatically correcting to apply_on_array=False.
  warn(


## 3.4. Chunk Loading, Windowing, and Geometry

In [12]:
def load_moabb_chunk(dataset_name, subject_ids, dataset_kwargs):
    print(f"Loading MOABB chunk: dataset={dataset_name} subjects={subject_ids} kwargs={dataset_kwargs}")
    dataset = MOABBDataset(
        dataset_name=dataset_name,
        subject_ids=[int(s) for s in subject_ids],
        dataset_kwargs=dict(dataset_kwargs),
    )
    if len(dataset.datasets) == 0:
        raise RuntimeError(f"No recordings loaded for {dataset_name} subjects={subject_ids}")
    return dataset


def apply_preprocessing_pipeline(dataset, preprocessors, dataset_name, subject_ids):
    with mne.use_log_level("WARNING"):
        preprocess(dataset, preprocessors, n_jobs=1)
    first_raw = dataset.datasets[0].raw
    print(
        f"  Preprocessed {dataset_name} subjects={subject_ids}: "
        f"recordings={len(dataset.datasets)} sfreq={first_raw.info['sfreq']} "
        f"channels={len(first_raw.ch_names)}"
    )


def create_pretraining_windows(dataset, window_size_samples, sampling_interval_samples, preload=False):
    windows = create_fixed_length_windows(
        dataset,
        start_offset_samples=0,
        stop_offset_samples=None,
        window_size_samples=window_size_samples,
        window_stride_samples=sampling_interval_samples,
        drop_last_window=True,
        preload=preload,
        n_jobs=1,
        targets_from="metadata",
        last_target_only=True,
    )
    if len(windows) == 0:
        raise RuntimeError("Chunk produced zero pretraining windows.")
    x0 = windows[0][0]
    if x0.shape[-1] != window_size_samples:
        raise RuntimeError(f"Window length mismatch: got {x0.shape[-1]}, expected {window_size_samples}")
    return windows


def extract_eeg_geometry(raw):
    eeg_picks = mne.pick_types(raw.info, eeg=True, meg=False, stim=False, eog=False)
    ch_names = [raw.ch_names[idx] for idx in eeg_picks]
    coords = []
    valid = []
    for idx in eeg_picks:
        loc = raw.info["chs"][idx].get("loc", None)
        if loc is None or len(loc) < 3:
            xyz = np.zeros(3, dtype=np.float32)
        else:
            xyz = np.asarray(loc[:3], dtype=np.float32)
            if not np.all(np.isfinite(xyz)):
                xyz = np.zeros(3, dtype=np.float32)
        coords.append(xyz)
        valid.append(bool(np.linalg.norm(xyz) > 1e-8))
    coords = np.stack(coords).astype(np.float32)
    valid = np.asarray(valid, dtype=bool)
    return {
        "ch_names": ch_names,
        "coords": torch.tensor(coords, dtype=torch.float32),
        "valid_mask": torch.tensor(valid, dtype=torch.bool),
        "n_channels": len(ch_names),
        "valid_ratio": float(valid.mean()) if len(valid) else 0.0,
    }


def prepare_pretraining_chunk_dataset(chunk, split_name):
    raw_dataset = load_moabb_chunk(
        dataset_name=chunk["dataset_name"],
        subject_ids=chunk["subject_ids"],
        dataset_kwargs=chunk["dataset_kwargs"],
    )
    apply_preprocessing_pipeline(
        dataset=raw_dataset,
        preprocessors=_PREPROCESSORS,
        dataset_name=chunk["dataset_name"],
        subject_ids=chunk["subject_ids"],
    )
    geometry = extract_eeg_geometry(raw_dataset.datasets[0].raw)
    windows = create_pretraining_windows(
        raw_dataset,
        window_size_samples=WINDOW_SIZE_SAMPLES,
        sampling_interval_samples=SAMPLING_INTERVAL_SAMPLES,
        preload=CONFIG["window_preload"],
    )
    print(
        f"  Windows ready: split={split_name} dataset={chunk['dataset_name']} "
        f"subjects={chunk['subject_ids']} windows={len(windows)} "
        f"shape={tuple(windows[0][0].shape)} valid_coord_ratio={geometry['valid_ratio']:.3f}"
    )
    return windows, raw_dataset, geometry


def make_chunk_dataloader(windows_dataset, batch_size, training):
    return DataLoader(
        windows_dataset,
        batch_size=batch_size,
        shuffle=bool(training),
        num_workers=0,
        drop_last=bool(training),
    )

## 3.5. Reference Chunk Sanity Check

In [ ]:
REFERENCE_CHUNK = TRAIN_CHUNKS_PREVIEW[0]
REFERENCE_WINDOWS_DATASET, REFERENCE_RAW_DATASET, REFERENCE_GEOMETRY = prepare_pretraining_chunk_dataset(
    REFERENCE_CHUNK,
    split_name="reference",
)

sample_x = REFERENCE_WINDOWS_DATASET[0][0]
print("\nReference chunk summary:")
print(f"  Dataset:              {REFERENCE_CHUNK['dataset_name']}")
print(f"  Subjects:             {REFERENCE_CHUNK['subject_ids']}")
print(f"  Window count:         {len(REFERENCE_WINDOWS_DATASET)}")
print(f"  Sample shape:         {tuple(sample_x.shape)}")
print(f"  Channels:             {REFERENCE_GEOMETRY['n_channels']}")
print(f"  Valid coord ratio:    {REFERENCE_GEOMETRY['valid_ratio']:.3f}")
print(f"  First 10 channels:    {REFERENCE_GEOMETRY['ch_names'][:10]}")

[2026-05-05 21:48:30] Loading MOABB chunk: dataset=Lee2019_MI subjects=[1, 2] kwargs={}


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 's3.ap-northeast-1.wasabisys.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
100%|████████████████████████████████████████| 621M/621M [00:00<00:00, 675GB/s]
SHA256 hash of downloaded file: 2f281b03f5493251855d65c4da107bd38ff1540aca0ea4b308b420c21bb89c66
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 's3.ap-northeast-1.wasabisys.com'. Adding certificate ver

[2026-05-05 21:50:29]   Preprocessed Lee2019_MI subjects=[1, 2]: recordings=4 sfreq=128.0 channels=62
[2026-05-05 21:50:29]   Windows ready: split=reference dataset=Lee2019_MI subjects=[1, 2] windows=1434 shape=(62, 512) valid_coord_ratio=1.000

[2026-05-05 21:50:29] Reference chunk summary:
[2026-05-05 21:50:29]   Dataset:              Lee2019_MI
[2026-05-05 21:50:29]   Subjects:             [1, 2]
[2026-05-05 21:50:29]   Window count:         1434
[2026-05-05 21:50:29]   Sample shape:         (62, 512)
[2026-05-05 21:50:29]   Channels:             62
[2026-05-05 21:50:29]   Valid coord ratio:    1.000
[2026-05-05 21:50:29]   First channels:       ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2']


# 4. Graph-SJEPA Model

## 4.1. Graph Utilities

In [14]:
def normalize_coords(coords, valid_mask=None, eps=1e-6):
    coords = coords.float()
    if valid_mask is None or bool(valid_mask.sum() == 0):
        valid = torch.ones(coords.shape[0], dtype=torch.bool, device=coords.device)
    else:
        valid = valid_mask.to(coords.device).bool()
    center = coords[valid].mean(dim=0, keepdim=True)
    centered = coords - center
    scale = centered[valid].norm(dim=1).max().clamp_min(eps)
    return centered / scale


def build_knn_adjacency(coords, k=6, self_loops=True, distance_weighted=True, eps=1e-6):
    """Build symmetric normalized adjacency from channel coordinates."""
    coords = normalize_coords(coords, eps=eps)
    n = coords.shape[0]
    if n < 2:
        return torch.eye(n, dtype=torch.float32, device=coords.device)

    dist = torch.cdist(coords, coords, p=2)
    k_eff = int(max(1, min(k, n - 1)))
    nn_idx = torch.topk(dist, k=k_eff + 1, largest=False).indices[:, 1:]

    adj = torch.zeros(n, n, dtype=torch.float32, device=coords.device)
    src = torch.arange(n, device=coords.device).unsqueeze(1).expand(-1, k_eff)
    if distance_weighted:
        weights = torch.exp(-dist[src, nn_idx])
    else:
        weights = torch.ones_like(dist[src, nn_idx])
    adj[src, nn_idx] = weights
    adj = torch.maximum(adj, adj.T)

    if self_loops:
        adj = adj + torch.eye(n, dtype=torch.float32, device=coords.device)

    deg = adj.sum(dim=1).clamp_min(eps)
    deg_inv_sqrt = torch.pow(deg, -0.5)
    adj_norm = deg_inv_sqrt[:, None] * adj * deg_inv_sqrt[None, :]
    return adj_norm


def expand_channel_mask_to_token_mask(mask_ch, n_tokens_per_channel):
    return mask_ch[:, :, None].expand(-1, -1, n_tokens_per_channel).reshape(mask_ch.shape[0], -1)

## 4.2. Spatial Block Masking

In [15]:
def sample_coordinate_spatial_block_mask(
    coords,
    batch_size,
    diameter_percent,
    min_visible_channels=2,
    device="cpu",
):
    """Sample one coordinate-radius channel mask per example."""
    coords = normalize_coords(coords.to(device))
    n_ch = coords.shape[0]
    if n_ch < 2:
        raise ValueError("Need at least 2 channels for spatial masking.")

    dist = torch.cdist(coords, coords, p=2)
    head_diameter = float(dist.max().detach().cpu())
    radius = (float(diameter_percent) / 100.0) * head_diameter / 2.0

    masks = torch.zeros(batch_size, n_ch, dtype=torch.bool, device=device)
    centers = torch.randint(low=0, high=n_ch, size=(batch_size,), device=device)

    for b in range(batch_size):
        center = int(centers[b].item())
        mask = dist[center] <= radius

        # Keep at least one masked channel.
        if int(mask.sum().item()) == 0:
            mask[center] = True

        # Keep enough visible channels for the graph path.
        visible_count = int((~mask).sum().item())
        if visible_count < min_visible_channels:
            # Unmask the farthest channels from the center until enough are visible.
            farthest = torch.argsort(dist[center], descending=True)
            for idx in farthest:
                mask[int(idx.item())] = False
                if int((~mask).sum().item()) >= min_visible_channels:
                    break

        masks[b] = mask

    return {
        "mask_ch": masks,
        "centers": centers,
        "head_diameter": head_diameter,
        "mask_radius": radius,
        "mean_masked_channels": float(masks.sum(dim=1).float().mean().detach().cpu()),
        "mean_visible_channels": float((~masks).sum(dim=1).float().mean().detach().cpu()),
    }


_mask_debug = sample_coordinate_spatial_block_mask(
    coords=REFERENCE_GEOMETRY["coords"],
    batch_size=8,
    diameter_percent=CONFIG["mask_diameter_percent"],
    min_visible_channels=CONFIG["min_visible_channels"],
    device="cpu",
)
print("Mask sanity check:")
print(f"  mean masked channels:  {_mask_debug['mean_masked_channels']:.2f}")
print(f"  mean visible channels: {_mask_debug['mean_visible_channels']:.2f}")
print(f"  mask radius:           {_mask_debug['mask_radius']:.4f}")

[2026-05-05 21:50:29] Mask sanity check:
[2026-05-05 21:50:29]   mean masked channels:  9.12
[2026-05-05 21:50:29]   mean visible channels: 52.88
[2026-05-05 21:50:29]   mask radius:           0.5173


## 4.3. Local Encoder

In [16]:
class ChannelLocalEncoder(nn.Module):
    """Encode each EEG channel independently into temporal latent tokens."""

    def __init__(self, emb_dim, conv_channels, kernel_size=7, stride=2, dropout=0.1):
        super().__init__()
        layers = []
        in_ch = 1
        for hidden in conv_channels:
            layers.extend([
                nn.Conv1d(
                    in_channels=in_ch,
                    out_channels=hidden,
                    kernel_size=kernel_size,
                    stride=stride,
                    padding=kernel_size // 2,
                    bias=False,
                ),
                nn.BatchNorm1d(hidden),
                nn.GELU(),
                nn.Dropout(dropout),
            ])
            in_ch = hidden
        layers.append(nn.Conv1d(in_ch, emb_dim, kernel_size=1, stride=1))
        self.net = nn.Sequential(*layers)
        self.emb_dim = int(emb_dim)

    def forward(self, x):
        # x: [B, C, T]
        b, c, t = x.shape
        x_flat = x.reshape(b * c, 1, t)
        z = self.net(x_flat)               # [B*C, D, Lt]
        z = z.transpose(1, 2)              # [B*C, Lt, D]
        z = z.reshape(b, c, z.shape[1], z.shape[2])
        return z                           # [B, C, Lt, D]

## 4.4. Coordinate, Graph, and Context Modules

In [17]:
class CoordinateEncoder(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        )

    def forward(self, coords):
        coords = normalize_coords(coords)
        return self.net(coords.float())


class GraphConvBlock(nn.Module):
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.lin_self = nn.Linear(emb_dim, emb_dim)
        self.lin_neigh = nn.Linear(emb_dim, emb_dim)
        self.norm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, adj):
        # x: [B, C, Lt, D]
        b, c, lt, d = x.shape
        h = x.permute(0, 2, 1, 3).reshape(b * lt, c, d)  # [B*Lt, C, D]
        neigh = torch.einsum("ij,bjd->bid", adj, h)
        out = self.lin_self(h) + self.lin_neigh(neigh)
        out = self.norm(F.gelu(out))
        out = self.dropout(out)
        out = out.reshape(b, lt, c, d).permute(0, 2, 1, 3)
        return out


class GraphSpatialEncoder(nn.Module):
    def __init__(self, emb_dim, n_layers=2, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([GraphConvBlock(emb_dim, dropout=dropout) for _ in range(n_layers)])

    def forward(self, x, adj):
        for layer in self.layers:
            x = x + layer(x, adj)
        return x


class GraphSJEPABackbone(nn.Module):
    def __init__(
        self,
        emb_dim=64,
        local_conv_channels=(32, 64, 64),
        local_kernel_size=7,
        local_stride=2,
        graph_layers=2,
        context_layers=4,
        context_nhead=8,
        context_dim_feedforward=256,
        dropout=0.1,
        max_tokens_per_channel=256,
    ):
        super().__init__()
        self.local_encoder = ChannelLocalEncoder(
            emb_dim=emb_dim,
            conv_channels=list(local_conv_channels),
            kernel_size=local_kernel_size,
            stride=local_stride,
            dropout=dropout,
        )
        self.coord_encoder = CoordinateEncoder(emb_dim)
        self.time_pos = nn.Parameter(torch.zeros(1, 1, max_tokens_per_channel, emb_dim))
        nn.init.trunc_normal_(self.time_pos, std=0.02)
        self.graph_encoder = GraphSpatialEncoder(emb_dim, n_layers=graph_layers, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=context_nhead,
            dim_feedforward=context_dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.context_encoder = nn.TransformerEncoder(enc_layer, num_layers=context_layers)
        self.final_norm = nn.LayerNorm(emb_dim)
        self.emb_dim = int(emb_dim)

    def forward(self, x, coords, adj, mask_ch=None):
        # x: [B, C, T]
        tokens = self.local_encoder(x)  # [B, C, Lt, D]
        b, c, lt, d = tokens.shape

        if lt > self.time_pos.shape[2]:
            raise RuntimeError(
                f"n_tokens_per_channel={lt} exceeds max_tokens_per_channel={self.time_pos.shape[2]}. "
                "Increase max_tokens_per_channel."
            )

        coord_pos = self.coord_encoder(coords.to(x.device)).view(1, c, 1, d)
        time_pos = self.time_pos[:, :, :lt, :]
        tokens = tokens + coord_pos + time_pos

        if mask_ch is not None:
            visible = (~mask_ch).float().to(x.device).view(b, c, 1, 1)
            tokens = tokens * visible

        tokens = self.graph_encoder(tokens, adj.to(x.device))
        seq = tokens.reshape(b, c * lt, d)
        seq = self.context_encoder(seq)
        seq = self.final_norm(seq)
        return seq, {"n_channels": c, "n_tokens_per_channel": lt, "total_tokens": c * lt}


class MaskedTokenPredictor(nn.Module):
    def __init__(self, emb_dim=64, n_layers=2, nhead=8, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.mask_token = nn.Parameter(torch.zeros(1, 1, emb_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.net = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.out = nn.Linear(emb_dim, emb_dim)

    def forward(self, context, mask_tok):
        x = context.clone()
        x = torch.where(mask_tok[:, :, None], x + self.mask_token, x)
        x = self.net(x)
        return self.out(x)

## 4.5. Instantiate Student, Teacher, and Predictor

In [18]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# Estimate max token count from the reference batch.
_reference_x = torch.as_tensor(REFERENCE_WINDOWS_DATASET[0][0]).float().unsqueeze(0)
_tmp_local = ChannelLocalEncoder(
    emb_dim=CONFIG["emb_dim"],
    conv_channels=CONFIG["local_conv_channels"],
    kernel_size=CONFIG["local_kernel_size"],
    stride=CONFIG["local_stride"],
    dropout=CONFIG["dropout"],
)
with torch.no_grad():
    _tmp_tokens = _tmp_local(_reference_x)
N_TOKENS_PER_CHANNEL = int(_tmp_tokens.shape[2])
MAX_TOKENS_PER_CHANNEL = max(256, N_TOKENS_PER_CHANNEL + 8)
del _tmp_local, _tmp_tokens, _reference_x

STUDENT = GraphSJEPABackbone(
    emb_dim=CONFIG["emb_dim"],
    local_conv_channels=CONFIG["local_conv_channels"],
    local_kernel_size=CONFIG["local_kernel_size"],
    local_stride=CONFIG["local_stride"],
    graph_layers=CONFIG["graph_layers"],
    context_layers=CONFIG["context_layers"],
    context_nhead=CONFIG["context_nhead"],
    context_dim_feedforward=CONFIG["context_dim_feedforward"],
    dropout=CONFIG["dropout"],
    max_tokens_per_channel=MAX_TOKENS_PER_CHANNEL,
).to(DEVICE)

TEACHER = deepcopy(STUDENT).to(DEVICE)
for p in TEACHER.parameters():
    p.requires_grad = False
TEACHER.eval()

PREDICTOR = MaskedTokenPredictor(
    emb_dim=CONFIG["emb_dim"],
    n_layers=CONFIG["predictor_layers"],
    nhead=CONFIG["predictor_nhead"],
    dim_feedforward=CONFIG["predictor_dim_feedforward"],
    dropout=CONFIG["dropout"],
).to(DEVICE)

student_total, student_trainable = count_parameters(STUDENT)
pred_total, pred_trainable = count_parameters(PREDICTOR)
print("Graph-SJEPA model instantiated:")
print(f"  n_tokens_per_channel estimate: {N_TOKENS_PER_CHANNEL}")
print(f"  max_tokens_per_channel:        {MAX_TOKENS_PER_CHANNEL}")
print(f"  student params:                {student_total:,} total / {student_trainable:,} trainable")
print(f"  predictor params:              {pred_total:,} total / {pred_trainable:,} trainable")

/var/folders/7d/njk_0cn503z09dk98r0csptr0000gn/T/ipykernel_149/3646316043.py:81: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.context_encoder = nn.TransformerEncoder(enc_layer, num_layers=context_layers)


[2026-05-05 21:50:30] Graph-SJEPA model instantiated:
[2026-05-05 21:50:30]   n_tokens_per_channel estimate: 64
[2026-05-05 21:50:30]   max_tokens_per_channel:        256
[2026-05-05 21:50:30]   student params:                285,472 total / 285,472 trainable
[2026-05-05 21:50:30]   predictor params:              104,192 total / 104,192 trainable


/var/folders/7d/njk_0cn503z09dk98r0csptr0000gn/T/ipykernel_149/3646316043.py:125: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.net = nn.TransformerEncoder(layer, num_layers=n_layers)


## 4.6. One-Batch Forward Sanity Check

In [19]:
def make_graph_inputs(geometry, device):
    coords = geometry["coords"].to(device)
    adj = build_knn_adjacency(
        coords,
        k=CONFIG["graph_k_neighbors"],
        self_loops=CONFIG["graph_self_loops"],
        distance_weighted=CONFIG["graph_distance_weighted"],
        eps=CONFIG["graph_eps"],
    ).to(device)
    return coords, adj


def masked_l1_loss(predicted, target, mask_tok):
    if mask_tok.sum() == 0:
        raise RuntimeError("No masked tokens selected for loss.")
    diff = torch.abs(predicted - target)
    return diff[mask_tok].mean()


def ema_update(student, teacher, decay):
    with torch.no_grad():
        for student_param, teacher_param in zip(student.parameters(), teacher.parameters()):
            teacher_param.data.mul_(decay).add_(student_param.data, alpha=1.0 - decay)


def run_graph_sjepa_batch(X, geometry, training=True):
    X = X.float().to(DEVICE)
    coords, adj = make_graph_inputs(geometry, DEVICE)
    mask_info = sample_coordinate_spatial_block_mask(
        coords=coords,
        batch_size=X.shape[0],
        diameter_percent=CONFIG["mask_diameter_percent"],
        min_visible_channels=CONFIG["min_visible_channels"],
        device=DEVICE,
    )
    mask_ch = mask_info["mask_ch"]

    student_context, token_info = STUDENT(X, coords=coords, adj=adj, mask_ch=mask_ch)
    with torch.no_grad():
        teacher_context, _ = TEACHER(X, coords=coords, adj=adj, mask_ch=None)

    mask_tok = expand_channel_mask_to_token_mask(mask_ch, token_info["n_tokens_per_channel"])
    pred = PREDICTOR(student_context, mask_tok=mask_tok)
    loss = masked_l1_loss(pred, teacher_context.detach(), mask_tok)

    return {
        "loss": loss,
        "loss_value": float(loss.detach().cpu()),
        "mask_info": mask_info,
        "token_info": token_info,
        "mean_masked_tokens": float(mask_tok.sum(dim=1).float().mean().detach().cpu()),
        "mean_visible_tokens": float((~mask_tok).sum(dim=1).float().mean().detach().cpu()),
    }

_debug_loader = make_chunk_dataloader(REFERENCE_WINDOWS_DATASET, batch_size=min(4, CONFIG["batch_size"]), training=False)
_debug_batch = next(iter(_debug_loader))[0]
_debug_result = run_graph_sjepa_batch(_debug_batch, REFERENCE_GEOMETRY, training=False)
print("One-batch Graph-SJEPA sanity check:")
print(f"  loss:                    {_debug_result['loss_value']:.6f}")
print(f"  token info:              {_debug_result['token_info']}")
print(f"  mean masked channels:    {_debug_result['mask_info']['mean_masked_channels']:.2f}")
print(f"  mean masked tokens:      {_debug_result['mean_masked_tokens']:.2f}")
print(f"  mean visible tokens:     {_debug_result['mean_visible_tokens']:.2f}")

if not np.isfinite(_debug_result["loss_value"]):
    raise RuntimeError("Sanity-check loss is non-finite.")

RuntimeError: MPS backend out of memory (MPS allocated: 18.06 GiB, other allocations: 1.90 GiB, max allowed: 20.13 GiB). Tried to allocate 1.88 GiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

# 5. Training

## 5.1. Optimizer and Batch Step

In [ ]:
OPTIMIZER = torch.optim.Adam(
    list(STUDENT.parameters()) + list(PREDICTOR.parameters()),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)


def pretraining_step(X, geometry):
    STUDENT.train()
    PREDICTOR.train()
    TEACHER.eval()

    OPTIMIZER.zero_grad(set_to_none=True)
    result = run_graph_sjepa_batch(X, geometry=geometry, training=True)
    loss = result["loss"]
    loss.backward()
    if CONFIG["grad_clip_norm"] is not None:
        torch.nn.utils.clip_grad_norm_(
            list(STUDENT.parameters()) + list(PREDICTOR.parameters()),
            max_norm=float(CONFIG["grad_clip_norm"]),
        )
    OPTIMIZER.step()
    ema_update(STUDENT, TEACHER, decay=CONFIG["ema_decay"])
    return result


@torch.no_grad()
def validation_step(X, geometry):
    STUDENT.eval()
    PREDICTOR.eval()
    TEACHER.eval()
    return run_graph_sjepa_batch(X, geometry=geometry, training=False)

## 5.2. Checkpoint Helpers

In [ ]:
def build_backbone_export_metadata():
    return {
        "model_type": "GraphSJEPABackbone",
        "run_id": RUN_ID,
        "config": CONFIG,
        "preprocessing": {
            "sfreq": CONFIG["sfreq"],
            "bandpass_low": CONFIG["bandpass_low"],
            "bandpass_high": CONFIG["bandpass_high"],
            "pretrain_duration_s": CONFIG["pretrain_duration_s"],
            "sampling_interval_s": CONFIG["sampling_interval_s"],
            "window_size_samples": WINDOW_SIZE_SAMPLES,
            "sampling_interval_samples": SAMPLING_INTERVAL_SAMPLES,
        },
        "graph": {
            "k_neighbors": CONFIG["graph_k_neighbors"],
            "self_loops": CONFIG["graph_self_loops"],
            "distance_weighted": CONFIG["graph_distance_weighted"],
        },
        "token_geometry_reference": {
            "n_tokens_per_channel": N_TOKENS_PER_CHANNEL,
            "max_tokens_per_channel": MAX_TOKENS_PER_CHANNEL,
            "emb_dim": CONFIG["emb_dim"],
        },
    }


BACKBONE_EXPORT_METADATA = build_backbone_export_metadata()


def save_checkpoint(tag, epoch, epoch_record, metrics_so_far):
    checkpoint = {
        "epoch": epoch,
        "epoch_record": epoch_record,
        "student_state_dict": STUDENT.state_dict(),
        "teacher_state_dict": TEACHER.state_dict(),
        "predictor_state_dict": PREDICTOR.state_dict(),
        "optimizer_state_dict": OPTIMIZER.state_dict(),
        "metrics": metrics_so_far,
        "backbone_export_metadata": BACKBONE_EXPORT_METADATA,
    }
    path = ARTIFACT_DIR / f"checkpoint_{tag}.pt"
    torch.save(checkpoint, path)
    return str(path)


def save_student_backbone_export(tag):
    payload = {
        "student_backbone_state_dict": STUDENT.state_dict(),
        "backbone_export_metadata": BACKBONE_EXPORT_METADATA,
    }
    path = ARTIFACT_DIR / f"student_graph_sjepa_backbone_{tag}.pt"
    torch.save(payload, path)
    return str(path)


def append_epoch_metrics(epoch_record):
    path = ARTIFACT_DIR / "epoch_metrics.jsonl"
    with open(path, "a") as f:
        f.write(json.dumps(epoch_record) + "\n")

## 5.3. Epoch Loop over Dataset/Subject Chunks

In [ ]:
def run_pretraining_epoch(epoch, split_name, training):
    epoch_start = time.time()
    chunks = iter_dataset_subject_chunks(
        split_name=split_name,
        shuffle=CONFIG["chunk_shuffle"] and training,
        seed=CONFIG["seed"] + int(epoch),
    )
    if len(chunks) == 0:
        raise RuntimeError(f"No chunks available for split={split_name}")

    batch_losses = []
    masked_channels = []
    visible_channels = []
    masked_tokens = []
    visible_tokens = []
    chunk_records = []

    for chunk_idx, chunk in enumerate(chunks, start=1):
        chunk_start = time.time()
        windows = None
        raw_dataset = None
        loader = None
        try:
            windows, raw_dataset, geometry = prepare_pretraining_chunk_dataset(chunk, split_name=split_name)
            loader = make_chunk_dataloader(windows, batch_size=CONFIG["batch_size"], training=training)
            chunk_losses = []

            for batch in loader:
                X = batch[0]
                if training:
                    result = pretraining_step(X, geometry)
                else:
                    result = validation_step(X, geometry)

                loss_value = float(result["loss_value"])
                chunk_losses.append(loss_value)
                batch_losses.append(loss_value)
                masked_channels.append(float(result["mask_info"]["mean_masked_channels"]))
                visible_channels.append(float(result["mask_info"]["mean_visible_channels"]))
                masked_tokens.append(float(result["mean_masked_tokens"]))
                visible_tokens.append(float(result["mean_visible_tokens"]))

            if len(chunk_losses) == 0:
                raise RuntimeError(
                    f"Chunk produced zero batches: split={split_name} dataset={chunk['dataset_name']} subjects={chunk['subject_ids']}"
                )

            chunk_record = {
                "dataset_name": chunk["dataset_name"],
                "subject_ids": chunk["subject_ids"],
                "n_recordings": len(raw_dataset.datasets),
                "n_windows": len(windows),
                "n_batches": len(chunk_losses),
                "mean_loss": float(np.mean(chunk_losses)),
                "time_s": round(time.time() - chunk_start, 2),
                "n_channels": int(geometry["n_channels"]),
                "coord_valid_ratio": float(geometry["valid_ratio"]),
            }
            chunk_records.append(chunk_record)

            print(
                f"[{split_name}] epoch={epoch} chunk={chunk_idx}/{len(chunks)} "
                f"dataset={chunk['dataset_name']} subjects={chunk['subject_ids']} "
                f"windows={len(windows)} batches={len(chunk_losses)} "
                f"channels={geometry['n_channels']} loss={chunk_record['mean_loss']:.6f} "
                f"running={float(np.mean(batch_losses)):.6f} time={chunk_record['time_s']}s"
            )

        finally:
            del loader, windows, raw_dataset
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if len(batch_losses) == 0:
        raise RuntimeError(f"Epoch produced zero batches for split={split_name}")

    return {
        "epoch": int(epoch),
        "split": split_name,
        "mean_loss": float(np.mean(batch_losses)),
        "min_loss": float(np.min(batch_losses)),
        "max_loss": float(np.max(batch_losses)),
        "n_batches": int(len(batch_losses)),
        "n_chunks": int(len(chunks)),
        "mean_masked_channels": float(np.mean(masked_channels)),
        "mean_visible_channels": float(np.mean(visible_channels)),
        "mean_masked_tokens": float(np.mean(masked_tokens)),
        "mean_visible_tokens": float(np.mean(visible_tokens)),
        "epoch_time_s": round(time.time() - epoch_start, 2),
        "chunks": chunk_records,
    }

## 5.4. Run Pretraining

In [ ]:
print("=" * 70)
print("STARTING GRAPH-SJEPA PRETRAINING")
print("=" * 70)
print(f"Datasets:             {[d['name'] for d in CONFIG['datasets']]}")
print(f"Train chunks/epoch:   {len(TRAIN_CHUNKS_PREVIEW)}")
print(f"Val chunks/epoch:     {len(VAL_CHUNKS_PREVIEW)}")
print(f"Window seconds:       {CONFIG['pretrain_duration_s']}")
print(f"Mask diameter %:      {CONFIG['mask_diameter_percent']}")
print(f"Device:               {DEVICE}")
print("=" * 70)

EPOCH_METRICS = []
BEST_VAL_LOSS = float("inf")
BEST_EPOCH = None
PATIENCE_COUNTER = 0
STOP_REASON = "max_epochs_reached"

for epoch in range(1, int(CONFIG["n_epochs"]) + 1):
    train_record = run_pretraining_epoch(epoch=epoch, split_name="train", training=True)
    val_record = run_pretraining_epoch(epoch=epoch, split_name="val", training=False)

    epoch_record = {
        "epoch": int(epoch),
        "train_loss": train_record["mean_loss"],
        "val_loss": val_record["mean_loss"],
        "train": train_record,
        "val": val_record,
    }
    EPOCH_METRICS.append(epoch_record)
    append_epoch_metrics(epoch_record)

    save_checkpoint("latest", epoch, epoch_record, EPOCH_METRICS)
    save_student_backbone_export("latest")

    improved = val_record["mean_loss"] < BEST_VAL_LOSS
    if improved:
        BEST_VAL_LOSS = val_record["mean_loss"]
        BEST_EPOCH = int(epoch)
        PATIENCE_COUNTER = 0
        save_checkpoint("best", epoch, epoch_record, EPOCH_METRICS)
        save_student_backbone_export("best")
        best_marker = "*"
    else:
        PATIENCE_COUNTER += 1
        best_marker = ""

    print(
        f"[epoch {epoch}] train={train_record['mean_loss']:.6f} "
        f"val={val_record['mean_loss']:.6f} best={BEST_VAL_LOSS:.6f} "
        f"patience={PATIENCE_COUNTER}/{CONFIG['early_stopping_patience']} {best_marker}"
    )

    if PATIENCE_COUNTER >= int(CONFIG["early_stopping_patience"]):
        STOP_REASON = "early_stopping"
        print(f"Early stopping triggered at epoch {epoch}.")
        break

print("Training loop finished.")

# 6. Diagnostics

## 6.1. Loss Curve

In [ ]:
if EPOCH_METRICS:
    epochs = [m["epoch"] for m in EPOCH_METRICS]
    train_losses = [m["train_loss"] for m in EPOCH_METRICS]
    val_losses = [m["val_loss"] for m in EPOCH_METRICS]

    plt.figure(figsize=(8, 4))
    plt.plot(epochs, train_losses, label="train")
    plt.plot(epochs, val_losses, label="validation")
    if BEST_EPOCH is not None:
        plt.axvline(BEST_EPOCH, linestyle="--", label=f"best epoch {BEST_EPOCH}")
    plt.xlabel("Epoch")
    plt.ylabel("Masked latent L1 loss")
    plt.title("Graph-SJEPA Pretraining Loss")
    plt.legend()
    plt.tight_layout()
    loss_plot_path = ARTIFACT_DIR / "loss_curve.png"
    plt.savefig(loss_plot_path, dpi=150)
    plt.show()
    print(f"Loss curve saved to: {loss_plot_path}")
else:
    print("No epoch metrics available for plotting.")

## 6.2. Graph and Mask Diagnostics

In [ ]:
def summarize_reference_graph_and_mask():
    coords = REFERENCE_GEOMETRY["coords"].to(DEVICE)
    adj = build_knn_adjacency(
        coords,
        k=CONFIG["graph_k_neighbors"],
        self_loops=CONFIG["graph_self_loops"],
        distance_weighted=CONFIG["graph_distance_weighted"],
        eps=CONFIG["graph_eps"],
    )
    mask_samples = sample_coordinate_spatial_block_mask(
        coords=coords,
        batch_size=256,
        diameter_percent=CONFIG["mask_diameter_percent"],
        min_visible_channels=CONFIG["min_visible_channels"],
        device=DEVICE,
    )
    return {
        "reference_dataset": REFERENCE_CHUNK["dataset_name"],
        "reference_subjects": REFERENCE_CHUNK["subject_ids"],
        "n_channels": int(REFERENCE_GEOMETRY["n_channels"]),
        "coord_valid_ratio": float(REFERENCE_GEOMETRY["valid_ratio"]),
        "adjacency_density": float((adj > 0).float().mean().detach().cpu()),
        "adjacency_min": float(adj.min().detach().cpu()),
        "adjacency_max": float(adj.max().detach().cpu()),
        "mask_radius": float(mask_samples["mask_radius"]),
        "mean_masked_channels": float(mask_samples["mean_masked_channels"]),
        "mean_visible_channels": float(mask_samples["mean_visible_channels"]),
    }

MASK_GRAPH_STATS = summarize_reference_graph_and_mask()
print(json.dumps(MASK_GRAPH_STATS, indent=2))

mask_graph_stats_path = ARTIFACT_DIR / "mask_graph_stats.json"
with open(mask_graph_stats_path, "w") as f:
    json.dump(MASK_GRAPH_STATS, f, indent=2)
print(f"Mask/graph stats saved to: {mask_graph_stats_path}")

## 6.3. One-Batch End-to-End Inspector

In [ ]:
@torch.no_grad()
def inspect_one_batch_end_to_end(windows_dataset, geometry):
    loader = make_chunk_dataloader(windows_dataset, batch_size=min(4, CONFIG["batch_size"]), training=False)
    X = next(iter(loader))[0]
    result = validation_step(X, geometry)
    return {
        "batch_shape": list(X.shape),
        "loss": result["loss_value"],
        "token_info": result["token_info"],
        "mean_masked_channels": result["mask_info"]["mean_masked_channels"],
        "mean_visible_channels": result["mask_info"]["mean_visible_channels"],
        "mean_masked_tokens": result["mean_masked_tokens"],
        "mean_visible_tokens": result["mean_visible_tokens"],
    }

INSPECTION = inspect_one_batch_end_to_end(REFERENCE_WINDOWS_DATASET, REFERENCE_GEOMETRY)
print(json.dumps(INSPECTION, indent=2))

inspection_path = ARTIFACT_DIR / "one_batch_inspection.json"
with open(inspection_path, "w") as f:
    json.dump(INSPECTION, f, indent=2)
print(f"One-batch inspection saved to: {inspection_path}")

# 7. Outputs

## 7.1. Save Metrics and Run Metadata

In [ ]:
metrics_path = ARTIFACT_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(EPOCH_METRICS, f, indent=2)
print(f"Epoch metrics saved to: {metrics_path}")

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "datasets": CONFIG["datasets"],
    "train_chunks_per_epoch": len(TRAIN_CHUNKS_PREVIEW),
    "val_chunks_per_epoch": len(VAL_CHUNKS_PREVIEW),
    "reference_chunk": REFERENCE_CHUNK,
    "reference_geometry": {
        "n_channels": int(REFERENCE_GEOMETRY["n_channels"]),
        "valid_ratio": float(REFERENCE_GEOMETRY["valid_ratio"]),
        "ch_names_preview": REFERENCE_GEOMETRY["ch_names"][:20],
    },
    "preprocessing": BACKBONE_EXPORT_METADATA["preprocessing"],
    "graph": BACKBONE_EXPORT_METADATA["graph"],
    "token_geometry_reference": BACKBONE_EXPORT_METADATA["token_geometry_reference"],
    "training": {
        "batch_size": CONFIG["batch_size"],
        "learning_rate": CONFIG["learning_rate"],
        "weight_decay": CONFIG["weight_decay"],
        "ema_decay": CONFIG["ema_decay"],
        "early_stopping_patience": CONFIG["early_stopping_patience"],
        "epochs_completed": len(EPOCH_METRICS),
        "best_epoch": BEST_EPOCH,
        "best_val_loss": BEST_VAL_LOSS,
        "stop_reason": STOP_REASON,
        "device": str(DEVICE),
    },
    "artifacts": {
        "config": str(config_path),
        "run_log": str(LOG_PATH),
        "metrics": str(metrics_path),
        "epoch_metrics_jsonl": str(ARTIFACT_DIR / "epoch_metrics.jsonl"),
        "checkpoint_latest": str(ARTIFACT_DIR / "checkpoint_latest.pt"),
        "checkpoint_best": str(ARTIFACT_DIR / "checkpoint_best.pt"),
        "student_graph_sjepa_backbone_latest": str(ARTIFACT_DIR / "student_graph_sjepa_backbone_latest.pt"),
        "student_graph_sjepa_backbone_best": str(ARTIFACT_DIR / "student_graph_sjepa_backbone_best.pt"),
        "mask_graph_stats": str(mask_graph_stats_path),
        "one_batch_inspection": str(inspection_path),
    },
}

metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(run_metadata, f, indent=2, default=str)
print(f"Run metadata saved to: {metadata_path}")

## 7.2. Experiment Summary

In [ ]:
print("\n" + "=" * 70)
print("GRAPH-SJEPA PRETRAINING SUMMARY")
print("=" * 70)
print(f"Run ID:               {RUN_ID}")
print(f"Artifacts:            {ARTIFACT_DIR}")
print()
print("Datasets:")
for ds_cfg in CONFIG["datasets"]:
    print(f"  {ds_cfg['name']}: role={ds_cfg.get('role')} train={ds_cfg['train_subject_ids']} val={ds_cfg['val_subject_ids']}")
print()
print("Model:")
print(f"  emb_dim:             {CONFIG['emb_dim']}")
print(f"  graph_layers:        {CONFIG['graph_layers']}")
print(f"  context_layers:      {CONFIG['context_layers']}")
print(f"  predictor_layers:    {CONFIG['predictor_layers']}")
print(f"  tokens/channel ref:  {N_TOKENS_PER_CHANNEL}")
print()
print("Training:")
print(f"  epochs completed:    {len(EPOCH_METRICS)}")
print(f"  best epoch:          {BEST_EPOCH}")
if BEST_VAL_LOSS < float("inf"):
    print(f"  best val loss:       {BEST_VAL_LOSS:.6f}")
else:
    print("  best val loss:       N/A")
print(f"  stop reason:         {STOP_REASON}")
print()
print("Best backbone export:")
print(f"  {ARTIFACT_DIR / 'student_graph_sjepa_backbone_best.pt'}")
print("=" * 70)